# Week 6 Student Worksheet: Spatial Prediction Shootout

## Traditional Stats vs. Machine Learning: Filling the Gaps

> *"Kriging tells you where it's uncertain. ML just says 'trust me.'"*

### Today's Mission

Transform Week 5's discrete rainfall stations into **continuous rainfall surfaces** using **two fundamentally different approaches**:

1. **Kriging (Statistical)** — uses spatial correlation + provides uncertainty
2. **Random Forest (ML)** — uses data patterns + easily adds features

Then compare them head-to-head with two simpler methods (Nearest Neighbor, IDW) and determine which gives the Commander more actionable intelligence.

> Fill in the code cells marked with `# YOUR CODE HERE`. Use AI tools strategically — but understand every line you write.

In [ ]:
# Install required packages (run once)
%pip install pykrige scikit-learn rasterio rasterstats --quiet

---

## Cell [1]: Environment Setup & Data Loading (Slide 2)

Load the Fung-wong typhoon data and filter to the study area. Reuse your Week 5 `parse_rainfall_json()` function.

**AI Prompt Suggestion**:
```
I need to load fungwong_202511.json and parse it into a GeoDataFrame using
my Week 5 parse_rainfall_json() function. Then filter to 花蓮縣 and 宜蘭縣,
remove -998/zero rain stations, and convert to EPSG:3826.
Show me the top 5 stations by rain_1hr.
```

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# YOUR CODE HERE:
# 1. Define normalize_cwa_json() and parse_rainfall_json() (copy from Week 5)
# 2. Load 'data/scenarios/fungwong_202511.json'
# 3. Parse into GeoDataFrame
# 4. Filter to 花蓮縣 + 宜蘭縣
# 5. Remove stations with rain_1hr <= 0
# 6. Convert to EPSG:3826


# Extract coordinate arrays for Kriging / ML
# x = study_rain_3826.geometry.x.values  # Easting (meters)
# y = study_rain_3826.geometry.y.values  # Northing (meters)
# z = study_rain_3826['rain_1hr'].values

# print(f"Study area stations (rain > 0): {len(study_rain_3826)}")
# print(f"CRS: {study_rain_3826.crs}")
# print(f"\nTop 5 stations:")
# print(study_rain_3826.nlargest(5, 'rain_1hr')[['station_name', 'county', 'rain_1hr']].to_string(index=False))

---

## Cell [2a]: Variogram — First Attempt (Naive)

Try running Kriging directly on the raw rainfall data. **Don't worry if it looks bad — that's the point.**

**Key Concepts** (will make sense after you see the result):
- **Nugget**: How noisy is each measurement?
- **Sill**: What's the maximum difference between stations?
- **Range**: How far apart do stations need to be before they stop "knowing about each other"?

**CRS Warning**: The x, y arrays MUST be in EPSG:3826 (meters). Using lat/lon will give wrong results.

In [ ]:
from pykrige.ok import OrdinaryKriging

# 🔴 First attempt: run Kriging on raw rainfall data
# YOUR CODE HERE:
# 1. Create OrdinaryKriging with raw z values
#    Use variogram_model='spherical', verbose=False, enable_plotting=True, nlags=15
# 2. Provide initial parameters to help the optimizer:
#    sill = z.var(), range = 50000, nugget = z.var() * 0.1

# initial_sill = float(z.var())
# initial_range = 50000.0
# initial_nugget = float(z.var() * 0.1)

# OK_naive = OrdinaryKriging(x, y, z, variogram_model='spherical',
#                             verbose=False, enable_plotting=True, nlags=15,
#                             variogram_parameters={'sill': initial_sill,
#                                                   'range': initial_range,
#                                                   'nugget': initial_nugget})

# params = OK_naive.variogram_model_parameters
# print(f"Sill:   {params[0]:.1f}")
# print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
# print(f"Nugget: {params[2]:.1f}")
# print("\n⚠️ Look at the plot — do the dots follow the curve?")

## Cell [2b]: Why Did It Fail? — Look at the Histogram

The variogram looked bad. Before blaming the tool, **look at your data**.

Plot a histogram of the rainfall values. What do you see?

In [ ]:
# YOUR CODE HERE:
# 1. Plot a histogram of z (raw rainfall)
# 2. Plot a histogram of np.log1p(z) (log-transformed)
# 3. Compare: which one looks more "balanced"?

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# axes[0].hist(z, bins=30, color='tomato', edgecolor='black')
# axes[0].set_title('Raw Rainfall (mm/hr)')
# axes[0].set_xlabel('Rainfall (mm/hr)')

# z_log = np.log1p(z)
# axes[1].hist(z_log, bins=30, color='steelblue', edgecolor='black')
# axes[1].set_title('After log(1+z)')
# axes[1].set_xlabel('log(1 + Rainfall)')
# plt.tight_layout()
# plt.show()

# print("Left: most stations < 10 mm, but a few are 50-130 mm.")
# print("Those extreme values mess up the variogram.")
# print("Right: after log-transform, the values are more balanced.")

## Cell [2c]: Variogram — Second Attempt (with Log-Transform)

Now redo the variogram using the log-transformed data. Compare this plot with Cell [2a] — it should be much better.

**Rule of thumb**: If your histogram has a long tail to the right, apply `np.log1p(z)` before Kriging, then use `np.expm1()` to convert back after prediction.

In [ ]:
# 🟢 Second attempt: Kriging on log-transformed data
# YOUR CODE HERE:
# 1. z_log = np.log1p(z)  (already computed above)
# 2. Create OrdinaryKriging with z_log (not z!)
# 3. Use initial parameters based on z_log.var()

# initial_sill = float(z_log.var())
# initial_range = 50000.0
# initial_nugget = float(z_log.var() * 0.1)

# OK = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
#                       verbose=False, enable_plotting=True, nlags=15,
#                       variogram_parameters={'sill': initial_sill,
#                                             'range': initial_range,
#                                             'nugget': initial_nugget})

# params = OK.variogram_model_parameters
# print(f"Sill:   {params[0]:.3f}")
# print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
# print(f"Nugget: {params[2]:.3f}")
# print("\n✅ Compare with Cell [2a] — the dots should follow the curve now.")

## Cell [2d]: Which Variogram Fits Best? — Range & Model Comparison (Slide 7)

Compare variograms **one variable at a time** — this is how scientists isolate effects:

- **Figure 1**: Fix model = **Spherical**, vary Range (50km, 25km, 15km) → what does Range do?
- **Figure 2**: Fix model = **Exponential**, vary Range (50km, 25km, 15km) → same exercise
- **Then compare**: same Range, different model → what does the model choice do?

**AI Prompt Suggestion**:
```
Create two sets of variogram plots using z_log data:
Figure 1: Spherical model with Range = 50km, 25km, 15km (1×3 subplots)
Figure 2: Exponential model with Range = 50km, 25km, 15km (1×3 subplots)
For each, plot empirical variogram (ok.lags vs ok.semivariance) as red dots
and the fitted model curve as a black line.
Compute SSE for each config and print a summary table.
Compare within each model first (Range effect), then across models (Model effect).
```

In [ ]:
# YOUR CODE HERE:
# 1. Define ranges: [50000, 25000, 15000]
# 2. Figure 1: Spherical × 3 Ranges (1×3 subplot)
#    For each: create OrdinaryKriging, plot lags vs semivariance (red dots),
#    plot fitted curve (black line), compute SSE
# 3. Figure 2: Exponential × 3 Ranges (1×3 subplot)
#    Same as above but with variogram_model='exponential'
# 4. Print summary table:
#    - Compare within Spherical (Range effect)
#    - Compare within Exponential (Range effect)
#    - Compare Spherical vs Exponential at same Range (Model effect)

# ranges_km = [50, 25, 15]

# ─── Figure 1: Spherical ───
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for ax, rkm in zip(axes, ranges_km):
#     ok_test = OrdinaryKriging(x, y, z_log, variogram_model='spherical', ...)
#     ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', ...)
#     # plot fitted curve ...

# ─── Figure 2: Exponential ───
# (same structure)

# 💡 Questions:
#   1. Within Spherical, which Range gives the best fit?
#   2. Within Exponential, which Range gives the best fit?
#   3. At the same Range, does model choice matter much?

## Cell [3]: Define the Interpolation Grid & Run Kriging (Slide 8)

Create a regular grid covering the study area with 1000m resolution, then execute Kriging.

**Think about**:
- What units is the grid in? (Hint: same as your CRS = meters)
- Why add a buffer around the station extent?

**Important**: Kriging runs in log-space. After prediction, **back-transform** with `np.expm1()` to get rainfall in mm/hr.

In [ ]:
import time

# YOUR CODE HERE:
# 1. Calculate grid extent from x, y arrays (with 5km buffer)
# 2. Create grid_x and grid_y using np.arange with 1000m step
# 3. Execute Kriging in log-space: z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
# 4. Back-transform: z_kriging = np.expm1(z_kriging_log)

buffer_m = 5000
resolution = 1000  # meters — use 500 for finer resolution (slower)

# x_min = x.min() - buffer_m
# x_max = x.max() + buffer_m
# y_min = y.min() - buffer_m
# y_max = y.max() + buffer_m
# grid_x = np.arange(x_min, x_max, resolution)
# grid_y = np.arange(y_min, y_max, resolution)

# print(f"Grid: {len(grid_x)}×{len(grid_y)} = {len(grid_x)*len(grid_y):,} points @ {resolution}m")

# t0 = time.time()
# z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
# print(f"✓ Kriging (log-space) done in {time.time()-t0:.1f}s")

# # Back-transform to real rainfall (mm/hr)
# z_kriging = np.expm1(z_kriging_log)
# z_kriging[z_kriging < 0] = 0
# ss_kriging = ss_kriging_log  # keep log-space variance for Sigma Map

# print(f"  z range (mm/hr): {np.nanmin(z_kriging):.1f} - {np.nanmax(z_kriging):.1f}")

---

## Cell [4]: Machine Learning — Random Forest Prediction (Slide 9)

**Captain's Log**: Treating coordinates as input features. ML learns `f(easting, northing) → rainfall`. Simple, but no uncertainty map.

**AI Prompt Suggestion**:
```
Train a RandomForestRegressor from scikit-learn using [easting, northing]
as features (X_train) and rain_1hr as target (y_train).
Use n_estimators=200, min_samples_leaf=3, random_state=42.
Then predict on the same grid as Kriging.
```

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# YOUR CODE HERE:
# 1. Prepare features: X_train = np.column_stack([x, y])
# 2. Train RandomForestRegressor
# 3. Create meshgrid from grid_x, grid_y
# 4. Predict on the grid

# X_train = np.column_stack([x, y])
# y_train = z

# rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
# rf.fit(X_train, y_train)
# print(f"RF Training R²: {rf.score(X_train, y_train):.3f}")

# grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
# X_grid = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])

# t0 = time.time()
# z_rf = rf.predict(X_grid).reshape(grid_xx.shape)
# print(f"✓ Random Forest done in {time.time()-t0:.1f}s")
# print(f"  z range: {z_rf.min():.1f} - {z_rf.max():.1f} mm/hr")

## Cell [5]: ML Glass Box — Feature Importance (Slide 11)

**Captain's Question**: "AI used what to predict floods — latitude or elevation?"

Even though ML is a "black box", we can peek inside with `.feature_importances_`.

In [ ]:
# YOUR CODE HERE:
# 1. Print rf.feature_importances_
# 2. Interpret: which dimension matters more for typhoon rainfall?

# importances = rf.feature_importances_
# print("Feature Importance:")
# print(f"  Easting (X):  {importances[0]:.3f}")
# print(f"  Northing (Y): {importances[1]:.3f}")
# print(f"\nThe model relies mostly on {'easting' if importances[0] > importances[1] else 'northing'}.")
# print("Think: does this make physical sense for Typhoon Fung-wong?")

---

## ⏸️ Lab 1: The Four-Way Interpolation Shootout (40 min)

### Cell [6]: Nearest Neighbor + IDW

Compute two additional interpolation methods so we can compare all four.

**AI Prompt Suggestion**:
```
Use scipy.interpolate.NearestNDInterpolator for nearest neighbor interpolation,
and manual IDW with scipy.spatial.distance.cdist (power=2) for IDW.
Both should predict on the same meshgrid as Kriging and RF.
Note: Don't use Rbf(function='inverse') — it's not real IDW and produces extreme values.
```

In [ ]:
from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist

# YOUR CODE HERE:
# 1. Nearest Neighbor interpolation
# 2. IDW interpolation (手動實作，power=2)
#    ⚠️ 注意：不要用 Rbf(function='inverse')，它不是真正的 IDW，會產生極端值

# nn_interp = NearestNDInterpolator(list(zip(x, y)), z)
# z_nn = nn_interp(grid_xx, grid_yy)

# pts = np.column_stack([x, y])
# grid_pts = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
# dists = cdist(grid_pts, pts)
# dists[dists < 1] = 1  # 避免除以零
# power = 2
# weights = 1.0 / (dists ** power)
# z_idw = ((weights @ z) / weights.sum(axis=1)).reshape(grid_xx.shape)

# print("✓ Nearest Neighbor + IDW computed")

### Cell [7]: Four Methods Side by Side (Slide 13)

Create a 2×2 figure comparing all four interpolation methods.

**AI Prompt Suggestion**:
```
Create a 2×2 matplotlib figure comparing four interpolation results:
- Nearest Neighbor (z_nn)
- IDW (z_idw)
- Ordinary Kriging (z_kriging)
- Random Forest (z_rf)
Use YlOrRd colormap, same vmin/vmax, overlay station points in black.
Add descriptive subtitles (e.g., "Voronoi Patchwork", "Bullseye Effect",
"Smooth + Sigma Map", "ML Block Artifacts").
Save as 'interpolation_shootout.png'.
```

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(2, 2, figsize=(18, 14))
# 2. Plot all four methods with imshow
# 3. Use extent=[x_min, x_max, y_min, y_max], origin='lower'
# 4. Overlay station scatter points
# 5. Add colorbars, titles, save figure

# vmax = max(z) * 1.1
# methods = [
#     ('Nearest Neighbor\n(Voronoi / "Patchwork")', z_nn),
#     ('IDW\n(Bullseye Effect)', z_idw),
#     ('Ordinary Kriging\n(Smooth + Sigma Map)', z_kriging),
#     ('Random Forest\n(ML "Block" Artifacts)', z_rf),
# ]

# for ax, (title, data) in zip(axes.flatten(), methods):
#     im = ax.imshow(data, extent=[x_min, x_max, y_min, y_max],
#                    origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
#     ax.scatter(x, y, c='black', s=8, zorder=5)
#     ax.set_title(title, fontsize=12, fontweight='bold')
#     plt.colorbar(im, ax=ax, shrink=0.7, label='mm/hr')

# plt.suptitle('Typhoon Fung-wong — Four Interpolation Methods Compared', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.savefig('interpolation_shootout.png', dpi=150, bbox_inches='tight')
# plt.show()

### Cell [8]: Kriging vs RF — Direct Comparison

Create a 3-panel comparison: Kriging | Random Forest | Difference Map

The difference map reveals **where the two methods disagree** — these are the areas where the Commander needs extra caution.

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(1, 3, figsize=(22, 7))
# 2. Left: Kriging (YlOrRd)
# 3. Middle: Random Forest (YlOrRd)
# 4. Right: Difference (Kriging - RF) using RdBu_r colormap
# 5. Save as 'kriging_vs_rf.png'

# diff = z_kriging - z_rf
# ...

### Lab 1 Reflection

**Questions to answer** (write in the cell below):

1. Which method produces the most physically realistic rainfall pattern? Why?
2. Where do Kriging and RF disagree the most? What does this tell you?
3. What "artifacts" do you observe in the NN and RF results?
4. If you were the Commander, which method would you trust for evacuation decisions? Why?

**Your Lab 1 reflection here:**

1. Most realistic: ...
2. Disagreement areas: ...
3. Artifacts: ...
4. Commander's choice: ...

---

## ⏸️ Lab 2: Confidence & Uncertainty Diagnosis (30 min)

### Cell [9]: The Sigma Map — Kriging's Unique Weapon (Slide 15)

**Captain's Log**: This is what makes Kriging special. No other method natively provides a confidence map.

- `ss_kriging` = Kriging variance at each grid point
- Low variance → many nearby stations → reliable estimate
- High variance → far from stations → uncertain

**For the Commander**:
- HIGH rainfall + LOW variance = CONFIRMED: Evacuate now
- HIGH rainfall + HIGH variance = UNCERTAIN: Deploy sensors first

In [ ]:
# YOUR CODE HERE:
# Create a 2-panel figure:
# Left: z_kriging with YlOrRd colormap (rainfall estimate)
# Right: ss_kriging with Blues colormap (variance/uncertainty)
# Add station locations on both panels (red dots on variance map)
# Save as 'sigma_map.png'

# fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# # Rainfall estimate
# im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
#                       origin='lower', cmap='YlOrRd', vmin=0)
# axes[0].scatter(x, y, c='black', s=10, zorder=5)
# axes[0].set_title('Estimated Rainfall (mm/hr)')
# plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# # Kriging Variance (Sigma Map)
# im2 = axes[1].imshow(ss_kriging, extent=[x_min, x_max, y_min, y_max],
#                       origin='lower', cmap='Blues', vmin=0)
# axes[1].scatter(x, y, c='red', s=10, zorder=5, label='Stations')
# axes[1].set_title('Kriging Sigma Map (Uncertainty)')
# axes[1].legend(loc='upper right')
# plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Variance')

# plt.suptitle('The Honest Report Card: Estimate + Confidence', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.savefig('sigma_map.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"Variance range: {np.nanmin(ss_kriging):.1f} - {np.nanmax(ss_kriging):.1f}")

### Cell [9b]: Nugget Effect — Why Extreme Rain Gets Diluted (Slide 7)

Suao recorded **130.5 mm/hr**, but default Kriging predicts only ~71 mm at 500m away. Why?

**Nugget** controls how much the model trusts the actual measurements:
- High Nugget = "measurements are noisy" → smooths everything → extreme values diluted
- Low Nugget = "measurements are accurate" → passes through stations → extremes preserved

**Task**: Compare Nugget=10% vs Nugget=1% on a zoomed-in map around Suao. Which preserves the extreme rainfall better?

**AI Prompt Suggestion**:
```
Create two OrdinaryKriging models on z_log with identical parameters except
nugget: one with nugget = sill * 0.10, one with nugget = sill * 0.01.
Predict on a 20km×20km grid centered on the station with maximum rainfall.
Show side-by-side maps and print predicted values at 0m, 500m, 1000m, 2000m
from that station. Which Nugget preserves the extreme value better?
```

In [ ]:
# YOUR CODE HERE:
# 1. Find the station with maximum rainfall (suao_idx = np.argmax(z))
# 2. Create two OrdinaryKriging models: nugget = sill*0.10 and sill*0.01
# 3. Predict on a local grid (20km box) around that station
# 4. Plot side-by-side comparison maps
# 5. Predict at specific offsets: 0m, 500m, 1000m, 2000m from the station
# 6. Print comparison table

# suao_idx = np.argmax(z)
# suao_x, suao_y, suao_z = x[suao_idx], y[suao_idx], z[suao_idx]
# print(f"Station with max rainfall: {suao_z:.1f} mm/hr")

# sill_val = float(z_log.var())
# ... create OK with nugget = sill_val * 0.10
# ... create OK with nugget = sill_val * 0.01
# ... compare predictions

# 🔑 Which Nugget setting is better for CWA calibrated stations? Why?

### Cell [10]: Export to GeoTIFF (Slide 16)

⚠️ **Flip warning**: `z_kriging` row 0 = south (numpy convention). GeoTIFF row 0 = north. Use `np.flipud()` to fix.

**AI Prompt Suggestion**:
```
Save 2D numpy arrays as GeoTIFF using rasterio. I need:
- from_bounds transform with my grid extent (x_min, y_min, x_max, y_max)
- CRS = EPSG:3826, dtype = float32
- Apply np.flipud() before writing
- Save kriging_rainfall.tif, kriging_variance.tif, and rf_rainfall.tif
```

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# YOUR CODE HERE:
# 1. Compute rasterio transform using from_bounds
# 2. Write a helper function save_geotiff(data, filename)
# 3. Save kriging_rainfall.tif, kriging_variance.tif, rf_rainfall.tif
# Remember: np.flipud() before writing!

# transform = from_bounds(x_min, y_min, x_max, y_max,
#                         width=z_kriging.shape[1], height=z_kriging.shape[0])

# def save_geotiff(data, filename, crs='EPSG:3826'):
#     data_flipped = np.flipud(data).astype(np.float32)
#     with rasterio.open(filename, 'w', driver='GTiff',
#         height=data_flipped.shape[0], width=data_flipped.shape[1],
#         count=1, dtype='float32', crs=crs, transform=transform, nodata=-9999
#     ) as dst:
#         dst.write(data_flipped, 1)
#     print(f"✓ Saved {filename}")

# save_geotiff(z_kriging, 'kriging_rainfall.tif')
# save_geotiff(ss_kriging, 'kriging_variance.tif')
# save_geotiff(z_rf, 'rf_rainfall.tif')

### Cell [11]: Zonal Statistics — Township Decision Table

Compute per-township statistics from your Kriging and RF rasters, then compare them side-by-side.

**Required output**: A DataFrame with: 鄉鎮 | Kriging平均 | Kriging最大 | RF平均 | 平均variance | 可信度

In [ ]:
from rasterstats import zonal_stats

# YOUR CODE HERE:
# 1. Load township boundaries (TGOS shapefile)
# 2. Filter to 花蓮縣 + 宜蘭縣, convert to EPSG:3826
# 3. Run zonal_stats on kriging_rainfall.tif, kriging_variance.tif, rf_rainfall.tif
# 4. Create summary DataFrame with columns:
#    鄉鎮, 縣市, Kriging平均, Kriging最大, RF平均, 平均variance
# 5. Add 可信度 column:
#    HIGH: variance < 33rd percentile
#    MEDIUM: 33rd-66th percentile
#    LOW: > 66th percentile

# Note: If you don't have the township shapefile, skip this cell
# and describe what the expected output would be in the markdown below.

# try:
#     towns = gpd.read_file('path/to/TOWN_MOI.shp')
#     study_towns = towns[towns['COUNTYNAME'].isin(['花蓮縣', '宜蘭縣'])].copy()
#     study_towns = study_towns.to_crs(epsg=3826)
#     ... (compute zonal stats and create summary table)
# except Exception as e:
#     print(f"Township shapefile not found: {e}")

### Cell [12]: Thinking Challenge — Why Can't ML Give You a Sigma Map?

**Discussion Questions** (answer in the cell below):

1. Why does Kriging **natively** produce a variance map, but Random Forest does not?
2. Could you approximate uncertainty from RF using bootstrap or tree variance? What are the limitations?
3. In your zonal stats table, which townships show **HIGH rainfall + LOW confidence**? What should the Commander do about those?

**Your Lab 2 reflection here:**

1. Kriging vs ML uncertainty: ...
2. ML uncertainty approximation: ...
3. High rain + low confidence townships: ...

### Cell [13]: (Bonus) AI Advisor Consultation

Ask an AI model (Gemini, ChatGPT, or Claude):

> 「在花蓮山區，測站密度約 1 站 / 50 km²。我用 Kriging 和 Random Forest 分別做了降雨內插，結果在山區差異很大。Kriging 的 variance 在山區也很高。請問：(1) 我應該信哪個結果？(2) 如何改善山區的預測品質？」

Paste the AI's response below and write your own commentary.

**AI Response:**

(Paste here)

**My Commentary:**

(Write 2-3 sentences on whether you agree with the AI's advice)

---

## ARIA Evolution Recap

| Version | Week | Capability | Key Question |
|---------|------|-----------|---------------|
| v1.0 | W3 | River buffer + shelters | How close to the river? |
| v2.0 | W4 | + DEM terrain analysis | How steep is the terrain? |
| v3.0 | W5 | + Real-time rainfall stations | How much rain RIGHT NOW? |
| **v3.5** | **W6** | **+ Kriging & ML interpolation** | **What about areas with NO station? How confident are we?** |
| v4.0 | Final Project | Your extension! | Your question! |

---

*"Interpolation is not just filling space; it is predicting risk where sensors cannot reach."*